In [15]:
import pandas as pd

train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

In [16]:
import numpy as np
import re
import torch
from datasets import Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, DataCollatorWithPadding
from scipy.special import softmax

In [17]:
conflicts = train_df[train_df.duplicated(['text'], keep=False)].groupby('text')['target'].nunique()
train_df = train_df[~train_df['text'].isin(conflicts[conflicts > 1].index)].drop_duplicates(subset=['text']).copy()

def minimal_preprocess(text):
    text = re.sub(r'http\S+', 'HTTPURL', text)
    text = re.sub(r'@\w+', '@USER', text)
    return re.sub(r'\s+', ' ', text).strip()

train_df['text'] = train_df['text'].apply(minimal_preprocess)
test_df['text'] = test_df['text'].apply(minimal_preprocess)

In [18]:
MODEL_NAME = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, normalization=True)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds)}

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
test_tokenized = Dataset.from_pandas(test_df[['id', 'text']]).map(tokenize_function, batched=True)


skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
test_probs = np.zeros((len(test_df), 2))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

training_args = TrainingArguments(
    output_dir="./results_bertweet_smoothing",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    label_smoothing_factor=0.1,  # Redukcja kary za brak absolutnej pewności
    weight_decay=0.01,
    warmup_steps=100,
    eval_strategy="epoch",
    save_strategy="no",
    seed=42,
    report_to="none"
)

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df['target'])):
    print(f"\n--- Trenowanie Foldu {fold + 1}/3 ---")

    train_fold = train_df.iloc[train_idx][['text', 'target']].rename(columns={'target': 'label'})
    val_fold = train_df.iloc[val_idx][['text', 'target']].rename(columns={'target': 'label'})

    tokenized_train_fold = Dataset.from_pandas(train_fold).map(tokenize_function, batched=True)
    tokenized_val_fold = Dataset.from_pandas(val_fold).map(tokenize_function, batched=True)

    model_fold = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

    trainer_fold = Trainer(
        model=model_fold,
        args=training_args,
        train_dataset=tokenized_train_fold,
        eval_dataset=tokenized_val_fold,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )

    trainer_fold.train()

    # Zbieranie ułamkowych przewidywań (Softmax) dla zbioru testowego
    preds_output = trainer_fold.predict(test_tokenized)
    test_probs += softmax(preds_output.predictions, axis=1) / skf.n_splits


[transformers] emoji is not installed, thus not converting emoticons or emojis into text. Install emoji: pip3 install emoji==0.6.0


Map:   0%|          | 0/3263 [00:00<?, ? examples/s]


--- Trenowanie Foldu 1/3 ---


Map:   0%|          | 0/4990 [00:00<?, ? examples/s]

Map:   0%|          | 0/2495 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

... (10 outputs left out)

Map:   0%|          | 0/2495 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/bertweet-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,No log,0.470598,0.828858,0.797918
2,0.485729,0.468405,0.841683,0.804165
3,0.485729,0.496409,0.846493,0.812531


In [19]:
final_preds = np.argmax(test_probs, axis=1)
submission = pd.DataFrame({'id': test_df['id'], 'target': final_preds})
submission.to_csv('submission_bertweet_label_smoothing.csv', index=False)